In [ ]:
import pandas as pd
import numpy as np

# Machine learning pipeline
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
#from xgboost import XGBClassifier

from sklearn.metrics import f1_score

In [ ]:
#train_values
train_values = pd.read_csv(r"C:\Users\olami\Downloads\train_values.csv")
train_values

In [ ]:
#train_labels
train_labels = pd.read_csv(r"C:\Users\olami\Downloads\train_labels.csv")
train_labels

In [ ]:
#-----------Understanding the data----------#

print('Count of target variables:')
print(train_labels.damage_grade.value_counts()) #Count of target variables

print('\nCount of feature values:') #Count of feature values
print(train_values.nunique())

print('\nData type of features and target:')
print(train_values.dtypes)#gets the data type of each column

print('\nColumn names:')
print(train_values.columns) #get Column names

print("\nNull values?", train_values.isnull().values.all()) #checks if the data has null values
#print(train_data.isnull().sum())

print("\nSummary statistics of the data:")
train_values.describe() #gets summary statistics of the data

No missing data and values match the data description

In [ ]:
#--------Separate categorical and numeric features so they can be preprocessed-----------#

building_id = 'building_id'
train_features = train_values.columns.drop(building_id) #dropping the building id as the model cannot learn anything from it


#numeric variables
NUMERICAL = train_values[train_features].select_dtypes('number').columns
print(f"Numerical features: {', '.join(NUMERICAL)}")

#categorical variables
CATEGORICAL = pd.Index(np.setdiff1d(train_features, NUMERICAL))
print(f"\nCategorical features: {', '.join(CATEGORICAL)}")

In [ ]:
#-------------Split the dataset into Train and Validation set--------------#

SEED = 42

X_train, X_validation, y_train, y_validation = train_test_split(train_values, train_labels['damage_grade'], 
                                                                test_size=.2, random_state=SEED, 
                                                                stratify=train_labels['damage_grade'])

In [ ]:
#------------Preprocessing and Model Pipeline-----------#

numerical_pipe = Pipeline([ ('imputer', SimpleImputer(strategy='mean')),
                            ('scaler', MinMaxScaler())
                         ])

categorical_pipe = Pipeline([ ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                              ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False))
                         ])

preprocessors = ColumnTransformer(transformers=[ ('num', numerical_pipe, NUMERICAL),
                                                 ('cat', categorical_pipe, CATEGORICAL)
                                 ])

pipe = Pipeline([ ('preprocessors', preprocessors),
                  ('model', LogisticRegression(max_iter=1000))
                ])

In [ ]:
#train Model

pipe.fit(X_train, y_train)

In [ ]:
#--------Performance Scoring---------#

y_pred = pipe.predict(X_validation)

f1_score(y_validation, y_pred, average='micro')

In [ ]:
pipe.score(X_validation, y_validation)

In [ ]:
#---------Load Test dataset----------#

test = pd.read_csv(r"C:\Users\olami\Downloads\test_values.csv")
test

In [ ]:
Pred_test = pipe.predict(test)

In [ ]:
# #------------Prediction submission for DrivenData------------#

# Pred_test_df = pd.DataFrame(Pred_test) #prediction to DataFrame
# submission = pd.concat([test['building_id'], Pred_test_df], axis=1)

In [ ]:
# submission.rename(columns = {0:'damage_grade'}, inplace=True)
# submission.to_csv(r"C:\Users\olami\Downloads\submission2.csv", index=False)

In [ ]:
# sub = pd.read_csv(r"C:\Users\olami\Downloads\submission2.csv")
# sub